In [53]:
# Problem Statement : Build a model to predict whether the mail is ham or spam

In [54]:
import pandas as pd
import numpy as np

In [55]:
df = pd.read_csv(r'C:\Users\Admin\Documents\NLP_Practice_Dataset\spam1 (1).csv',encoding="cp1252")

# The dataset is readed using encoding="cp1252" because it contains characters encoded in windows 1252 format.
# Using this code prevents UnicodeDecodeError and ensures that special characters are correctly interpreted during data loading

In [56]:
df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
6771,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
6772,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
6773,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
6774,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [57]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [58]:
df.shape

# Rows  = 6776
# Columns = 5

(6776, 5)

In [59]:
# since there are multiple features containing  only nan values so keeping those columns wont 
# make any sense so we need to remove those columns 

df = df.iloc[:,[0,1]]

In [60]:
df.head()

# So now we have only 1 feature and 1 target variable

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [61]:
# Now we need to rename the feature to x whereas target to y 

import warnings 
warnings.filterwarnings('ignore')

df.rename(columns = {'v1':'y','v2':'x'},inplace =True)


In [62]:
# verfiying the names of the target and fetaures

df.columns

Index(['y', 'x'], dtype='object')

In [63]:
# Lets check the number of classes in the target variable 

df['y'].unique()

# So there are 2 classes 'ham' and 'spam'

array(['ham', 'spam'], dtype=object)

In [64]:
# Since the machines cant understand the text language so we need to convert it into numbers 

df['y'].replace({'ham':0,'spam':1},inplace = True)

In [65]:
df['y'].unique()

# successfully  replaced the classes values with 0 and 1 

array([0, 1])

In [90]:
df['y'].value_counts()

y
0    5854
1     922
Name: count, dtype: int64

# Text Preprocessing 

In [66]:
# Coverting all  the text to lower as its case sensitive

df['x'].str.lower()

0       go until jurong point, crazy.. available only ...
1                           ok lar... joking wif u oni...
2       free entry in 2 a wkly comp to win fa cup fina...
3       u dun say so early hor... u c already then say...
4       nah i don't think he goes to usf, he lives aro...
                              ...                        
6771    this is the 2nd time we have tried 2 contact u...
6772                will ì_ b going to esplanade fr home?
6773    pity, * was in mood for that. so...any other s...
6774    the guy did some bitching but i acted like i'd...
6775                           rofl. its true to its name
Name: x, Length: 6776, dtype: object

In [67]:
# importing stopwords 
from nltk.corpus import stopwords

In [68]:
stopwords.words('english')

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [69]:
# lets check total number of stopwords

len(stopwords.words('english'))

198

In [70]:
import string 


In [71]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

# Before building the model we need to remove stopwords and punctuations

In [72]:
def text_process(mess):
    nopunc = [i for i in mess if i not in string.punctuation]
    nopunc = "".join(nopunc)
    return [i for i in nopunc.split() if i not in stopwords.words('english')]

In [73]:
df['x'].apply(text_process)

0       [Go, jurong, point, crazy, Available, bugis, n...
1                          [Ok, lar, Joking, wif, u, oni]
2       [Free, entry, 2, wkly, comp, win, FA, Cup, fin...
3           [U, dun, say, early, hor, U, c, already, say]
4       [Nah, I, dont, think, goes, usf, lives, around...
                              ...                        
6771    [This, 2nd, time, tried, 2, contact, u, U, å£7...
6772             [Will, Ì, b, going, esplanade, fr, home]
6773                     [Pity, mood, Soany, suggestions]
6774    [The, guy, bitching, I, acted, like, id, inter...
6775                              [Rofl, Its, true, name]
Name: x, Length: 6776, dtype: object

In [74]:
# Since we are done with stopwords and punctuation removal 
# Now we need to convert the text into numbers which can be understand by machines

from sklearn.feature_extraction.text import CountVectorizer

In [75]:
var1 = CountVectorizer(analyzer = text_process).fit(df.x)

In [76]:
len(var1.vocabulary_)


11480

In [77]:
tdm = var1.transform(df.x)

In [78]:
tdm.shape

(6776, 11480)

# Here tdm will act as you independent variable 

# Train Test split 

In [79]:
from sklearn.model_selection import train_test_split

tdm_train,tdm_test,y_train,y_test = train_test_split(tdm,df.y,test_size = 0.2)



# Model Building

In [98]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()

# fitting the model 

logreg.fit(tdm_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [99]:
pred = logreg.predict(tdm_test)

In [100]:
from sklearn.metrics import confusion_matrix,classification_report

tab = confusion_matrix(y_test,pred)

In [101]:
tab


array([[1176,    3],
       [  21,  156]])

In [102]:
print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1179
           1       0.98      0.88      0.93       177

    accuracy                           0.98      1356
   macro avg       0.98      0.94      0.96      1356
weighted avg       0.98      0.98      0.98      1356



In [85]:
# Lets use Decision tree algorithm and evaluate the performance between the models 

from sklearn.tree import DecisionTreeClassifier

DT = DecisionTreeClassifier()

# fitting the model 
DT.fit(tdm_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [86]:
# Creating a variable of prediction 
pred = DT.predict(tdm_test)


In [89]:
# Evaluating the model performance 

from sklearn.metrics import confusion_matrix,classification_report 

cm = confusion_matrix(y_test,pred)
cr = classification_report(y_test,pred)

print(cm)
print(cr)

[[1172    7]
 [  24  153]]
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1179
           1       0.96      0.86      0.91       177

    accuracy                           0.98      1356
   macro avg       0.97      0.93      0.95      1356
weighted avg       0.98      0.98      0.98      1356



# Logistic Regression with class_weigt = 'balanced'

In [104]:
# Since the class is imbalance we will handle the imbalance and check the models performance 

# creating an object 
lreg = LogisticRegression(class_weight = 'balanced')

# fitting the model 
lreg.fit(tdm_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [105]:
# creating an object of prediction 
pred = lreg.predict(tdm_test)

In [106]:
# evaluating the performance of the logistic regression model 

lr_cm = confusion_matrix(y_test,pred)
lr_cr = classification_report(y_test,pred)

print(lr_cm)
print(lr_cr)

[[1174    5]
 [  16  161]]
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1179
           1       0.97      0.91      0.94       177

    accuracy                           0.98      1356
   macro avg       0.98      0.95      0.96      1356
weighted avg       0.98      0.98      0.98      1356



# Random Forest witrh class_weight = 'balanced'

In [132]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(class_weight = 'balanced')

model.fit(tdm_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [133]:
# creating an object of prediction 

pred = model.predict(tdm_test)

In [134]:
# Evaluating the performance of the model 

rf_cm = confusion_matrix(y_test,pred)
rf_cr = classification_report(y_test,pred)

print(rf_cm)
print(rf_cr)

[[1179    0]
 [  32  145]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      1179
           1       1.00      0.82      0.90       177

    accuracy                           0.98      1356
   macro avg       0.99      0.91      0.94      1356
weighted avg       0.98      0.98      0.98      1356



# Using SMOTE 

In [135]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state = 42)

tdm_train_res,y_train_res = smote.fit_resample(tdm_train,y_train)

# Decision Tree

In [136]:
# Training the model 

model  = DecisionTreeClassifier(random_state = 42)

# fitting the model 

model.fit(tdm_train_res,y_train_res)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [138]:
# creating an object of prediction

pred = model.predict(tdm_test)

In [140]:
# Evaluating the performance of the model

dt_cm = confusion_matrix(y_test,pred)
dt_cr = classification_report(y_test,pred)

print(dt_cm)
print(dt_cr)

[[1059  120]
 [  16  161]]
              precision    recall  f1-score   support

           0       0.99      0.90      0.94      1179
           1       0.57      0.91      0.70       177

    accuracy                           0.90      1356
   macro avg       0.78      0.90      0.82      1356
weighted avg       0.93      0.90      0.91      1356



# Random Forest

In [144]:
model = RandomForestClassifier()

model.fit(tdm_train_res,y_train_res)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [145]:
pred = model.predict(tdm_test)

In [146]:
ra_cf = confusion_matrix(y_test,pred)
ra_cr = classification_report(y_test,pred)

print(ra_cf)
print(ra_cr)

[[1097   82]
 [  12  165]]
              precision    recall  f1-score   support

           0       0.99      0.93      0.96      1179
           1       0.67      0.93      0.78       177

    accuracy                           0.93      1356
   macro avg       0.83      0.93      0.87      1356
weighted avg       0.95      0.93      0.94      1356



# Logistic Regression 

In [151]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(tdm_train_res,y_train_res)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [152]:
pred = model.predict(tdm_test)

In [155]:
log_r_cm = confusion_matrix(y_test,pred)
log_r_cr = classification_report(y_test,pred)

print(log_r_cm)
print(log_r_cr)

[[1106   73]
 [  11  166]]
              precision    recall  f1-score   support

           0       0.99      0.94      0.96      1179
           1       0.69      0.94      0.80       177

    accuracy                           0.94      1356
   macro avg       0.84      0.94      0.88      1356
weighted avg       0.95      0.94      0.94      1356



# Conclusion

In [ ]:
# Multiple models were evaluated for the spam classification task, including Logistic Regression, Decision Tree, and Random Forest, with different 
# techniques to handle class imbalance such as SMOTE and class weighting.

# The baseline Logistic Regression and Decision Tree models (without imbalance handling) performed well in terms of accuracy (~98%), 
# but showed relatively lower recall for the minority class (spam), indicating missed spam instances.

# Applying SMOTE improved recall for the minority class across models; however, it significantly reduced precision, leading to a higher number 
# of false positives. This effect was more pronounced in Decision Tree and Random Forest models, where precision dropped substantially 
# despite improvements in recall.

# Using class weighting (class_weight='balanced') with Logistic Regression provided the best overall performance. It achieved high accuracy (~98%), 
# strong recall (~0.91) for the spam class, and excellent precision (~0.97), maintaining a good balance between detecting spam and 
# minimizing false positives.

# Random Forest with class weighting also performed well but did not outperform Logistic Regression in terms of overall balance between precision
# and recall.

# Final Model Selection:
# The Logistic Regression model with class_weight='balanced' is selected as the final model because it provides the best trade-off between precision
# and recall, effectively identifying spam emails while minimizing incorrect classifications.